# **Modelo Computacional con Redes Neuronales**
## **1. Librerías**
Las librerías $csv$, $random$, $math$, $json$ y $time$ en Python se utilizan para tareas comunes de programación: $csv$ permite leer y escribir archivos CSV, facilitando el manejo de datos tabulares; $random$ se usa para generar números aleatorios y realizar selecciones aleatorias, útil en simulaciones y algoritmos estocásticos; $math$ proporciona funciones matemáticas básicas y avanzadas como trigonometría, logaritmos y raíces; $json$ permite codificar y decodificar datos en formato JSON, muy utilizado para el intercambio de información; y $time$ permite medir tiempos de ejecución, pausar procesos y trabajar con fechas y horas en un nivel básico.

In [1]:


import csv
import random
import math
import json
import time




## **2. Carga y limpieza de datos**
La función $load\_data(filename)$ lee un archivo CSV delimitado por punto y coma ($;$), omite el encabezado si existe, filtra filas malformadas o con datos faltantes (indicados como $-99.9$), convierte los valores de cada fila a tipos numéricos apropiados (enteros y flotantes), y almacena los datos válidos en una lista que luego retorna; está diseñada para procesar registros diarios de variables como precipitación, temperatura y nivel de agua.

In [16]:
def load_data(filename):
    data = []
    with open(filename, newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=';')

        # Saltar encabezado si existe
        headers = next(reader)
        if headers[0].upper() == 'YEAR' or headers[0].isdigit():
            pass  # Encabezado detectado, ya se salto con next()

        for row in reader:
            if len(row) != 7:
                continue  # Saltar filas malformadas

            if '-99.9' in row:
                continue  # Saltar fila con dato faltante

            try:
                year = int(row[0])
                month = int(row[1])
                day = int(row[2])
                precip = float(row[3])
                maxtemp = float(row[4])
                mintemp = float(row[5])
                level = float(row[6])

                data.append([year, month, day, precip, maxtemp, mintemp, level])
            except ValueError as e:
                print(f"Error al procesar fila {row}: {e}")

    return data

In [25]:
import numpy as np
import pandas as pd
# en gurpos de 3, terniles
data = load_data("dataset.csv")
df = pd.read_csv("dataset.csv", sep=";")

# datos estaticos de LAKE_LEVEL
lake_level = df["LAKE_LEVEL"]
print(lake_level)

# separa de que rango a que rango estarian estas 3 etiquetas Bajo ,Medio, Alto
# imprime los intervalos de cada etiqueta (terciles por frecuencia: ~33% de datos en cada grupo)
cats, bordes = pd.qcut(lake_level, q=3, labels=["Bajo", "Medio", "Alto"], retbins=True, duplicates="drop")
etiquetas = ["Bajo", "Medio", "Alto"]
for i, nombre in enumerate(etiquetas):
    if i + 1 >= len(bordes):
        break
    lo, hi = bordes[i], bordes[i + 1]
    par_izq = "[" if i == 0 else "("
    print(f"{nombre}: {par_izq}{lo:.4f}, {hi:.4f}]  (mismo número aprox. de días en cada tercil)")


0        3810.35
1        3810.35
2        3810.35
3        3810.35
4        3810.35
          ...   
11189    3808.16
11190    3808.05
11191    3807.94
11192    3807.87
11193    3807.87
Name: LAKE_LEVEL, Length: 11194, dtype: float64
Bajo: [3807.8700, 3809.1800]  (mismo número aprox. de días en cada tercil)
Medio: (3809.1800, 3810.0900]  (mismo número aprox. de días en cada tercil)
Alto: (3810.0900, 3812.5400]  (mismo número aprox. de días en cada tercil)


La función $normalize(data)$ realiza una normalización min-max de los datos numéricos, es decir, escala cada valor de la matriz $data$ (lista de listas) al rango $[0, 1]$. Para ello, calcula los valores mínimos y máximos de cada columna usando $zip(*data)$, que permite trabajar columna por columna. Luego, recorre cada fila y aplica la fórmula de normalización $(x-mín)/(máx - mín)$ a cada valor, sumando una pequeña constante $1e-8$ al denominador para evitar divisiones por cero. Devuelve una nueva matriz con los datos normalizados, junto con las listas de mínimos y máximos originales, que pueden usarse posteriormente para revertir la normalización o aplicar la misma escala a nuevos datos.

In [3]:
def normalize(data):
    mins = [min(col) for col in zip(*data)]
    maxs = [max(col) for col in zip(*data)]

    normalized = []
    for row in data:
        norm_row = [(x - mn) / (mx - mn + 1e-8) for x, mn, mx in zip(row, mins, maxs)]
        normalized.append(norm_row)
    return normalized, mins, maxs

In [5]:
raw_data = load_data("dataset.csv")
normalized_data, mins, maxs = normalize(raw_data)


## **3. División de los datos**
La función $train\_test\_split(data, test\_size=0.2)$ divide un conjunto de datos en dos partes: una para entrenamiento y otra para prueba. Calcula el índice de corte multiplicando la longitud del conjunto de datos por $1-test\_size$, que por defecto es $0.2$ (es decir, $80\%$ para entrenamiento y $20\%$ para prueba). Luego retorna dos subconjuntos: el primero con los datos desde el inicio hasta el índice de corte (conjunto de entrenamiento) y el segundo con el resto de los datos (conjunto de prueba), permitiendo evaluar el desempeño de modelos de manera más objetiva.

In [4]:
def train_test_split(data, test_size=0.2):
    split_idx = int(len(data) * (1 - test_size))
    return data[:split_idx], data[split_idx:]

## **4. Funciones de Activación**
Se implementan activaciones comunes utilizadas en redes neuronales artificiales. La función $sigmoid(x)$ aplica la activación sigmoide, que transforma cualquier valor real en un rango entre $0$ y $1$, útil para modelar probabilidades. Su derivada $dsigmoid(y)$ calcula la pendiente de la sigmoide en función del valor ya activado $y$, facilitando el cálculo del gradiente durante la retropropagación. La función $relu(x)$ aplica la activación $ReLU (Rectified Linear Unit)$, que devuelve 0 para valores negativos y el valor original para positivos, ayudando a reducir el problema del desvanecimiento del gradiente. Su derivada $drelu(y)$ retorna 1 si el valor activado es mayor que 0 y 0 en caso contrario, siendo esencial en el entrenamiento de redes profundas.









In [5]:
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def dsigmoid(y):
    return y * (1 - y)

def relu(x):
    return max(0, x)

def drelu(y):
    return 1 if y > 0 else 0

## **5. Clase Neuron**
La clase $Neuron$ define una neurona artificial simple que puede utilizar funciones de activación $sigmoid$, $relu$ o lineal. En el constructor __init__, se inicializan los pesos y el sesgo (bias) con valores aleatorios pequeños, y se guarda la función de activación elegida.
El método $activate(inputs)$ calcula la suma ponderada de las entradas más el sesgo, y luego aplica la función de activación correspondiente (sigmoid, ReLU o lineal).
El método $derivative(output)$ devuelve la derivada de la función de activación evaluada en la salida ya activada, lo cual es fundamental para el cálculo del gradiente durante el entrenamiento mediante retropropagación. Esta clase representa la unidad básica de procesamiento en una red neuronal.

In [6]:
class Neuron:
    def __init__(self, inputs, activation="sigmoid"):
        self.weights = [random.uniform(-0.1, 0.1) for _ in range(inputs)]
        self.bias = random.uniform(-0.1, 0.1)
        self.activation_fn = activation

    def activate(self, inputs):
        weighted_sum = sum(w * i for w, i in zip(self.weights, inputs)) + self.bias
        if self.activation_fn == "sigmoid":
            return sigmoid(weighted_sum)
        elif self.activation_fn == "relu":
            return relu(weighted_sum)
        else:
            return weighted_sum

    def derivative(self, output):
        if self.activation_fn == "sigmoid":
            return dsigmoid(output)
        elif self.activation_fn == "relu":
            return drelu(output)
        else:
            return 1

El código define una clase $Layer$ que representa una capa de una red neuronal artificial compuesta por múltiples neuronas. En el constructor ($\_\_init\_\_$), se crean tantas neuronas como se indique con $n\_neurons$, y cada una recibe $n\_inputs$ entradas y utiliza una función de activación especificada (por defecto, "sigmoid"). El método $forward$ implementa la propagación hacia adelante: toma un conjunto de entradas y devuelve una lista con las salidas de cada neurona al activar dichas entradas. Para funcionar correctamente, este código asume la existencia previa de una clase $Neuron$ que incluya el método $activate$.

In [7]:
class Layer:
    def __init__(self, n_inputs, n_neurons, activation="sigmoid"):
        self.neurons = [Neuron(n_inputs, activation) for _ in range(n_neurons)]

    def forward(self, inputs):
        return [neuron.activate(inputs) for neuron in self.neurons]

La clase $NeuralNetwork$ implementa una red neuronal con una capa oculta y una capa de salida, diseñada para tareas de regresión (dado que la capa de salida usa activación lineal). En el constructor, se instancian las capas ($input\_layer$ y $output\_layer$) con el tamaño y función de activación especificados, y se define la tasa de aprendizaje $lr$. El método $forward$ realiza la propagación hacia adelante, primero a través de la capa oculta y luego la de salida. El método $backward$ implementa la retropropagación del error: calcula los errores en la salida, propaga el error hacia la capa oculta, y actualiza los pesos y sesgos de ambas capas usando el gradiente descendente. Finalmente, el método $train$ entrena la red a lo largo de múltiples épocas, calculando el error cuadrático medio (MSE) en cada iteración y aplicando $forward$ y $backward$ para ajustar los parámetros. Para que funcione, se requiere una clase $Neuron$ que implemente $activate$, $derivative$, $weights$ y $bias$.

In [8]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01, activation="sigmoid"):
        self.input_layer = Layer(input_size, hidden_size, activation)
        self.output_layer = Layer(hidden_size, output_size, "linear")
        self.lr = lr

    def forward(self, inputs):
        h_output = self.input_layer.forward(inputs)
        return self.output_layer.forward(h_output)

    def backward(self, inputs, target, output):
        # Backpropagation
        output_error = [t - o for t, o in zip(target, output)]

        # Output layer deltas
        o_deltas = [output_error[i] * out_neuron.derivative(output[i]) for i, out_neuron in enumerate(self.output_layer.neurons)]

        # Hidden layer deltas
        h_errors = [sum(o_deltas[j] * out_neuron.weights[i] for j, out_neuron in enumerate(self.output_layer.neurons))
            for i in range(len(self.input_layer.neurons))]
        h_deltas = [h_err * hid_neuron.derivative(hid_neuron.activate(inputs))
                    for i, (h_err, hid_neuron) in enumerate(zip(h_errors, self.input_layer.neurons))]

        # Update weights and biases
        for out_neuron, delta in zip(self.output_layer.neurons, o_deltas):
            for i in range(len(out_neuron.weights)):
                out_neuron.weights[i] += self.lr * delta * self.input_layer.forward(inputs)[i]
            out_neuron.bias += self.lr * delta

        for hid_neuron, delta in zip(self.input_layer.neurons, h_deltas):
            for i in range(len(hid_neuron.weights)):
                hid_neuron.weights[i] += self.lr * delta * inputs[i]
            hid_neuron.bias += self.lr * delta

    def train(self, X_train, y_train, epochs=100):
        for epoch in range(epochs):
            total_loss = 0
            for x, y in zip(X_train, y_train):
                output = self.forward(x)
                loss = sum((y - o)**2 for y, o in zip([y], output))  # MSE
                total_loss += loss
                self.backward(x, [y], output)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(X_train):.8f}")

## **6. Entrenamiento, Predicción y Evaluación de 7 Modelos**
El código implementa un sistema completo para entrenar, predecir y evaluar siete redes neuronales con distintas configuraciones, con el objetivo de analizar cómo influyen diferentes parámetros en el rendimiento del modelo. La función $train\_seven\_models$ entrena siete instancias de la clase $NeuralNetwork$, variando elementos como la función de activación ($sigmoid$ o $ReLU$), el tamaño de la capa oculta (8, 10 o 12 neuronas), la tasa de aprendizaje ($lr$), el número de épocas de entrenamiento (100 o 200) e incluso menciona un modelo con regularización L2 (aunque esta regularización aún no se aplica explícitamente). Cada modelo se entrena con los mismos datos de entrada y salida ($X\_train$, $y\_train$), y se registra el tiempo de entrenamiento para fines comparativos.

Luego, la función $predict$ toma un modelo entrenado y genera predicciones para un conjunto de prueba ($X\_test$), extrayendo la única salida producida por cada red. La función $evaluate$ calcula múltiples métricas de regresión para comparar la calidad de las predicciones respecto a los valores reales ($y\_true$). Estas métricas incluyen: error absoluto medio (MAE), error cuadrático medio (MSE), raíz del error cuadrático medio (RMSE), el coeficiente de determinación R², la varianza explicada (EVS), el error máximo (MaxError) y el porcentaje de error absoluto medio (MAPE). Este conjunto de funciones permite no solo construir y entrenar modelos personalizados, sino también compararlos cuantitativamente para elegir la mejor arquitectura y configuración de red neuronal.

In [9]:
def train_seven_models(X_train, y_train):
    models = []
    training_times = []

    # Modelo 1 - Sigmoid
    start = time.time()
    m1 = NeuralNetwork(input_size=6, hidden_size=16, output_size=1, lr=0.005, activation="relu")
    m1.train(X_train, y_train, epochs=800)
    end = time.time()
    models.append(m1)
    training_times.append(end - start)

    # Modelo 2 - ReLU
    start = time.time()
    m2 = NeuralNetwork(input_size=6, hidden_size=20, output_size=1, lr=0.008, activation="relu")
    m2.train(X_train, y_train, epochs=600)
    end = time.time()
    models.append(m2)
    training_times.append(end - start)

    # Modelo 3 - Tamaño diferente
    start = time.time()
    m3 = NeuralNetwork(input_size=6, hidden_size=14, output_size=1, lr=0.003, activation="sigmoid")
    m3.train(X_train, y_train, epochs=900)
    end = time.time()
    models.append(m3)
    training_times.append(end - start)

    # Modelo 4 - Tasa de aprendizaje diferente
    start = time.time()
    m4 = NeuralNetwork(input_size=6, hidden_size=16, output_size=1, lr=0.01, activation="relu")
    m4.train(X_train, y_train, epochs=500)
    end = time.time()
    models.append(m4)
    training_times.append(end - start)

    # Modelo 5 - Más épocas
    start = time.time()
    m5 = NeuralNetwork(input_size=6, hidden_size=18, output_size=1, lr=0.006, activation="relu")
    m5.train(X_train, y_train, epochs=1000)
    end = time.time()
    models.append(m5)
    training_times.append(end - start)

    # Modelo 6 - Regularización L2 (simulada)
    start = time.time()
    m6 = NeuralNetwork(input_size=6, hidden_size=12, output_size=1, lr=0.002, activation="relu")
    m6.train(X_train, y_train, epochs=850)
    end = time.time()
    models.append(m6)
    training_times.append(end - start)

    # Modelo 7 - Otra variante
    start = time.time()
    m7 = NeuralNetwork(input_size=6, hidden_size=24, output_size=1, lr=0.004, activation="relu")
    m7.train(X_train, y_train, epochs=750)
    end = time.time()
    models.append(m7)
    training_times.append(end - start)

    return models, training_times

def predict(model, X_test):
    return [model.forward(x)[0] for x in X_test]

def evaluate(y_true, y_pred):
    n = len(y_true)
    mae = sum(abs(t - p) for t, p in zip(y_true, y_pred)) / n
    mse = sum((t - p)**2 for t, p in zip(y_true, y_pred)) / n
    rmse = mse**0.5
    r2 = 1 - (sum((t - p)**2 for t, p in zip(y_true, y_pred)) /
               sum((t - sum(y_true)/n)**2 for t in y_true))
    evs = 1 - (sum((t - p)**2 for t, p in zip(y_true, y_pred)) /
                  sum((t - sum(y_true)/n)**2 for t in y_true))
    me = max(abs(t - p) for t, p in zip(y_true, y_pred))
    mape = sum(abs((t - p)/t) for t, p in zip(y_true, y_pred) if t != 0) / n

    return {
        "MAE": mae, #
        "MSE": mse, #
        "RMSE": rmse, #
        "R2": r2, #
        "EVS": evs, #
        "MaxError": me, #
        "MAPE": mape #
    }

## **7. Función Principal**
Se ejecuta todo el flujo de trabajo para entrenar y evaluar múltiples redes neuronales. Primero, carga un conjunto de datos desde un archivo CSV ($dataset.csv$) usando la función $load\_data$ y luego normaliza los datos con la función $normalize$, lo que asegura que todos los valores estén en un mismo rango (por ejemplo, entre 0 y 1). Después, separa las características de entrada ($X$, las primeras 6 columnas) y las salidas esperadas ($y$, la séptima columna).

A continuación, divide los datos en conjuntos de entrenamiento y prueba usando la función $train\_test\_split$. Luego, entrena siete modelos de redes neuronales con diferentes configuraciones usando $train\_seven\_models$, registrando el tiempo que tarda cada uno. Una vez entrenados los modelos, realiza predicciones sobre los datos de prueba ($X\_test$) con cada modelo y evalúa su rendimiento comparando las predicciones con las etiquetas verdaderas ($y\_test$) mediante la función $evaluate$.

Finalmente, imprime los tiempos de entrenamiento y las métricas de evaluación (como MAE, MSE, RMSE, R², entre otras) para cada modelo, y almacena estos resultados en una lista $results$. Este enfoque permite comparar de manera sistemática el impacto de diversas configuraciones en el desempeño del modelo.

In [11]:
if __name__ == "__main__":
    raw_data = load_data("dataset.csv")
    normalized_data, mins, maxs = normalize(raw_data)

    X = [row[:6] for row in normalized_data]
    y = [row[6] for row in normalized_data]

    X_train, X_test = train_test_split(X)
    y_train, y_test = train_test_split(y)

    models, training_times = train_seven_models(X_train, y_train)

    results = []
    for i, (model, train_time) in enumerate(zip(models, training_times)):
        preds = predict(model, X_test)
        metrics = evaluate(y_test, preds)

        print(f"\nModelo {i+1} - Tiempo de entrenamiento: {train_time:.2f} segundos")
        print(f"Modelo {i+1} - Métricas:")
        for key, val in metrics.items():
            print(f"{key:>10}: {val:.8f}")
        results.append(metrics)

Epoch 1/800, Loss: 0.00755250
Epoch 2/800, Loss: 0.00527819
Epoch 3/800, Loss: 0.00478608
Epoch 4/800, Loss: 0.00453168
Epoch 5/800, Loss: 0.00440835
Epoch 6/800, Loss: 0.00432427
Epoch 7/800, Loss: 0.00424767
Epoch 8/800, Loss: 0.00417600
Epoch 9/800, Loss: 0.00410708
Epoch 10/800, Loss: 0.00404218
Epoch 11/800, Loss: 0.00398805
Epoch 12/800, Loss: 0.00394028
Epoch 13/800, Loss: 0.00389646
Epoch 14/800, Loss: 0.00385450
Epoch 15/800, Loss: 0.00381523
Epoch 16/800, Loss: 0.00377666
Epoch 17/800, Loss: 0.00373903
Epoch 18/800, Loss: 0.00369992
Epoch 19/800, Loss: 0.00365879
Epoch 20/800, Loss: 0.00361636
Epoch 21/800, Loss: 0.00357151
Epoch 22/800, Loss: 0.00352330
Epoch 23/800, Loss: 0.00347109
Epoch 24/800, Loss: 0.00342022
Epoch 25/800, Loss: 0.00336253
Epoch 26/800, Loss: 0.00330194
Epoch 27/800, Loss: 0.00323158
Epoch 28/800, Loss: 0.00315853
Epoch 29/800, Loss: 0.00308674
Epoch 30/800, Loss: 0.00302059
Epoch 31/800, Loss: 0.00295594
Epoch 32/800, Loss: 0.00290157
Epoch 33/800, Los

El código implementa un sistema completo para almacenar, cargar y utilizar modelos de redes neuronales personalizadas entrenadas previamente para predecir el nivel del lago Titicaca a partir de datos meteorológicos. Primero, se define una función $save\_model$ que serializa los parámetros clave de cada modelo (pesos, sesgos, funciones de activación y tasa de aprendizaje) en un archivo JSON con extensión $.nn$. Luego, con $load\_model$, se reconstruye la red neuronal leyendo esos archivos y recreando capa por capa el modelo original con sus respectivos parámetros. Posteriormente, se pide al usuario que ingrese datos del día a predecir (año, mes, día, precipitación, temperatura máxima y mínima), los cuales son normalizados usando los valores mínimos y máximos del conjunto original de entrenamiento. Estos datos son ingresados en cada uno de los siete modelos guardados, y el resultado de la predicción es desnormalizado para obtener el valor real estimado del nivel del lago. Finalmente, el código muestra la predicción generada por cada modelo, permitiendo comparar sus resultados y observar el comportamiento del sistema ante nuevos datos de entrada.

In [ ]:
def save_model(model, filename):
    data = {
        "input_layer": {
            "neurons": [
                {"weights": neuron.weights, "bias": neuron.bias, "activation": neuron.activation_fn}
                for neuron in model.input_layer.neurons
            ]
        },
        "output_layer": {
            "neurons": [
                {"weights": neuron.weights, "bias": neuron.bias}
                for neuron in model.output_layer.neurons
            ]
        },
        "lr": model.lr
    }

    with open(filename, 'w') as f:
        json.dump(data, f)

def load_model(filename, input_size=6, output_size=1):
    with open(filename, 'r') as f:
        data = json.load(f)

    # Crear la red vacía
    input_neurons = []
    for neuron_data in data["input_layer"]["neurons"]:
        n = Neuron(input_size, activation=neuron_data["activation"])
        n.weights = neuron_data["weights"]
        n.bias = neuron_data["bias"]
        input_neurons.append(n)

    output_neurons = []
    for neuron_data in data["output_layer"]["neurons"]:
        n = Neuron(len(data["input_layer"]["neurons"]), activation="linear")
        n.weights = neuron_data["weights"]
        n.bias = neuron_data["bias"]
        output_neurons.append(n)

    # Reconstruir capas
    model = NeuralNetwork(input_size, len(input_neurons), output_size, lr=data["lr"])
    model.input_layer.neurons = input_neurons
    model.output_layer.neurons = output_neurons

    return model

# Después de entrenar los modelos
for i, model in enumerate(models):
    save_model(model, f"modelo_{i+1}.nn")

def predict_with_saved_model(model_index, input_data, mins, maxs):
    model = load_model(f"modelo_{model_index}.nn")

    # Normalizar entrada
    normalized_input = [(x - mn) / (mx - mn + 1e-8) for x, mn, mx in zip(input_data, mins, maxs[:-1])]

    prediction = model.forward(normalized_input)[0]
    # Desnormalizar salida
    prediction_denorm = prediction * (maxs[-1] - mins[-1]) + mins[-1]
    return prediction_denorm

def get_user_input():
    print("Ingrese los siguientes datos para predecir el nivel del lago:")

    year = int(input("Año (YEAR): "))
    month = int(input("Mes (MONTH): "))
    day = int(input("Día (DAY): "))
    precipitation = float(input("Precipitación (PRECIPITATION): "))
    max_temp = float(input("Temperatura máxima (MAXTEMPERATURE): "))
    min_temp = float(input("Temperatura mínima (MINTEMPERATURE): "))

    return [year, month, day, precipitation, max_temp, min_temp]

# Datos de entrada: YEAR, MONTH, DAY, PRECIPITATION, MAXTEMPERATURE, MINTEMPERATURE
#new_data = [2025, 4, 5, 0, 18, 5]
new_data = get_user_input()

# Realizar predicciones con los 7 modelos guardados
for i in range(1, 8):  # Modelos del 1 al 7
    predicted_level = predict_with_saved_model(i, new_data, mins, maxs)
    print(f"Modelo {i}: Predicción del nivel del lago: {predicted_level:.2f}")

Ingrese los siguientes datos para predecir el nivel del lago:


Año (YEAR):  2027
Mes (MONTH):  09
Día (DAY):  15


Se guarda y carga los valores mínimos y máximos usados para normalizar los datos en el entrenamiento del modelo. La función $save\_minmax$ recibe las listas $mins$ y $maxs$ con los valores mínimos y máximos de cada variable del dataset y los guarda en un archivo JSON llamado por defecto $minmax.json$. Luego, la función $load\_minmax$ lee ese archivo y devuelve esos valores, asegurando que la normalización y desnormalización de los datos nuevos o de prueba sean consistentes con el procesamiento original. Finalmente, el código guarda estos valores con $save\_minmax$ y los recarga con $load\_minmax$ para verificar que se hayan almacenado correctamente, imprimiéndolos en pantalla. Esto es fundamental para que las predicciones de los modelos sean precisas al aplicar la misma escala usada durante el entrenamiento.

In [6]:
def save_minmax(mins, maxs, filename="minmax.json"):
    data = {
        "mins": mins,
        "maxs": maxs
    }
    with open(filename, 'w') as f:
        json.dump(data, f)

def load_minmax(filename="minmax.json"):
    with open(filename, 'r') as f:
        data = json.load(f)
    return data["mins"], data["maxs"]

save_minmax(mins, maxs)

mins, maxs = load_minmax()
print(mins, maxs)

[1982, 1, 1, 0.0, 3.2, -7.0, 3808.12] [2012, 12, 31, 78.2, 22.8, 10.4, 3812.54]
